# Question answering system using RAG over books
This is the notebook version of RAG pipeline.

**Pipeline changes** (all in [rag_pipeline.py](rag_pipeline.py), each one a config flag):

| v2 |
|---|
| Mistral chat template, only new tokens decoded, greedy decoding |
| BGE-small dense + BM25, fused with Reciprocal Rank Fusion |
| Cross-encoder re-ranker (`BAAI/bge-reranker-base`) |
| Each chunk is indexed with a `Book: <title> by <author>` header |
| If the question names a book, search only that book |
| One abstention threshold on the re-ranker score, **calibrated on a dev split** |
| Numbered excerpts and `[n]` citations |

**Evaluation** (in [rag_evaluation.py](rag_evaluation.py)):
- **Test set:** 23 hand-checked questions with exact evidence quotes, ~60 synthetic questions spread over all 20 books, and 21 negatives (off-topic, books not in the library, false premises, gibberish). The split is stratified: thresholds are tuned on **dev** and results are reported on **test**.
- **Retrieval:** Hit@k, MRR, and nDCG@10 at the passage level (a hit means the retrieved chunk actually contains the gold evidence), plus how well the abstention score separates answerable from unanswerable questions (AUROC). Run as an ablation over 7 configurations.
- **End to end:** answer accuracy (LLM judge), token F1, faithfulness and hallucination rate (from claim-by-claim checks), citation precision, how often the system refuses answerable questions, and how often it correctly handles unanswerable ones. Also reported: latency p50/p95, 95% bootstrap confidence intervals, and a failure label for each question (retrieval miss / hallucination / generation error).

**Runtime:** about 45-60 minutes on a Colab L4 (most of it is the LLM judge). `MAX_TEST_ITEMS` below gives a faster run.

## 0) Setup

In [ ]:
!pip install -q -U langchain-text-splitters faiss-cpu sentence-transformers rank_bm25 langgraph datasets accelerate

In [ ]:
!ollama pull mistral
!pip install langchain-text-splitters faiss-cpu sentence-transformers rank_bm25 langgraph datasets accelerate pandas jupyter ollama

In [ ]:
# Load the helper modules. In Colab, clone the repo. Locally, run from this folder.
import os, sys
if not os.path.exists("rag_pipeline.py"):
    if not os.path.exists("LLM-Library-Minds"):
        !git clone -q https://github.com/Anvithanm/LLM-Library-Minds.git
    os.chdir("LLM-Library-Minds/RAG-llm-knowledge-integration")
sys.path.insert(0, os.getcwd())

import pandas as pd
pd.set_option("display.width", 200); pd.set_option("display.max_colwidth", 120)

from rag_pipeline import (RAGConfig, RAGPipeline, HFChatLLM, load_books, chunk_books, build_langgraph)
from rag_evaluation import (
    build_testset, generate_synthetic_testset, split_dev_test, IndexFactory, default_ablation_configs,
    compare_retrieval_configs, calibrate_abstention, evaluate_end_to_end, LLMJudge,
    comparison_table, qtype_table, save_results, show_failures, export_for_human_review, judge_human_agreement)

OUT_DIR = "eval_outputs"
os.makedirs(OUT_DIR, exist_ok=True)
MAX_TEST_ITEMS = None   # e.g. 30 for a quick run

Mistral-7B-Instruct is a gated model. Accept its license on Hugging Face and add an `HF_TOKEN` secret in Colab (key icon in the left sidebar).

## 1) Load the books

> **Data-quality note:** some dataset titles don't match the text. `"The Mysterious Island"` (Anthony Hope) is not Verne's novel, `"The Life of Julius Caesar"` (Melville) is *Bartleby, the Scrivener*, and `"The Lord of the Rings"` (Tolkien) is Machiavelli's *The Prince*. Apostrophes are also stripped (`won t`). The system answers with the dataset's labels, so a question like *"Who is Frodo in The Lord of the Rings?"* will retrieve Machiavelli.

In [ ]:
books = load_books(n_books=20)
pd.DataFrame([{k: b[k] for k in ("book_id", "title", "author", "category")} | {"chars": len(b["text"])} for b in books])

## 2) Load the LLM

The same wrapper is used for answering, writing synthetic questions, and judging.
For less biased judging, create a second `HFChatLLM` with a different model (e.g. `Qwen/Qwen2.5-7B-Instruct` with `load_in_4bit=True`, after `pip install bitsandbytes`) and pass it to `LLMJudge`. A model grading its own answers tends to be too lenient.

In [ ]:
llm = HFChatLLM("mistralai/Mistral-7B-Instruct-v0.3")
print(llm([{"role": "user", "content": "In one sentence, what is retrieval-augmented generation?"}], max_new_tokens=60))

In [3]:
import ollama
def llm(messages, max_new_tokens=256, temperature=0.0):
    r = ollama.chat(model="mistral", messages=messages,
                    options={"num_predict": max_new_tokens, "temperature": temperature})
    return r["message"]["content"].strip()

print(llm([{"role": "user", "content": "In one sentence, what is retrieval-augmented generation?"}], max_new_tokens=60))


Retrieval-augmented generation is a method in AI that combines a language model with a retrieval system to generate responses by both generating new content and recalling relevant information from a large dataset.

In simpler terms, it's a way for AI to answer questions by not


## 3) Build the test set

* **Curated** ([eval_data/curated_questions.json](eval_data/curated_questions.json)): hand-written questions whose evidence quotes were checked against the book text.
* **Synthetic:** chunks are sampled evenly across all 20 books (skipping front matter). The LLM writes a question, an answer, and an *exact* supporting quote. Items are dropped if the quote can't be found, if the question refers to "the passage", or if it duplicates another question. The quote's position in the book becomes the gold span. The set is cached in `eval_data/synthetic_questions.json`, so later runs reuse the same questions.
* **Negatives** ([eval_data/negative_questions.json](eval_data/negative_questions.json)): off-topic questions, books that are in the dataset but not indexed (Frankenstein, Monte Cristo...), false premises, and gibberish.

Add your own questions to the JSON files to grow the curated set. That makes the evaluation more reliable over time.

In [4]:
if "books" not in globals():
    books = load_books(n_books=20)

chunks = chunk_books(books, chunk_size=1000, chunk_overlap=200)
synthetic = generate_synthetic_testset(books, chunks, llm, n_questions=60,
                                       out_path="eval_data/synthetic_questions.json")
items = build_testset(books, synthetic)
dev, test = split_dev_test(items, dev_frac=0.3, seed=0)
if MAX_TEST_ITEMS:  # random sample so every question type stays represented
    import random
    test = random.Random(0).sample(test, min(MAX_TEST_ITEMS, len(test)))
print(f"{len(items)} questions -> dev {len(dev)} / test {len(test)}")
pd.crosstab(pd.Series([i["qtype"] for i in items], name="qtype"),
            pd.Series(["dev" if i in dev else "test" for i in items], name="split"))

Loaded 60 cached synthetic questions from eval_data/synthetic_questions.json
104 questions -> dev 31 / test 73


split,dev,test
qtype,,
cross_book,0,1
explanatory,1,2
factoid,5,12
false_premise,1,3
gibberish,1,2
metadata,1,1
missing_book,2,6
out_of_domain,2,4
synthetic,18,42


In [5]:
# Spot-check a few synthetic questions. Their quality limits how much the scores can be trusted.
pd.DataFrame(synthetic)[["title", "question", "reference_answer"]].sample(8, random_state=0)

,title,question,reference_answer
26,Great Expectations,Why does the lawyer in Great Expectations seem to have been engaged on a case of a darker complexion than usual?,The lawyer was engaged on a case of a darker complexion than usual because he was found with his head inside a close...
35,Robinson Crusoe,What animals were present during Crusoe's dinner?,A dog and two cats
59,Little Women,Why does Jo feel regret in this scene from 'Little Women'?,"Jo feels regret because she is disappointed about not being able to go somewhere, as mentioned by 'It is a dreadful ..."
28,The Adventures of Sherlock Holmes,Why did Holmes and Watson rush into the room after breaking the door?,The room was empty.
11,The Adventures of Sherlock Holmes,What is the husband's theory about his wife's behavior?,The husband thinks it's possible that the excitement of the affair caused a nervous disturbance in his wife.
2,The Mysterious Island,What is the reason for the disapprobation of the Kirton mob towards Medland?,"The Kirton mob would disapprove of Medland due to his involvement in a controversial fight with Kilshaw, as they wou..."
34,Pride and Prejudice,Why was Lady Catherine extremely indignant about the marriage of her nephew?,Lady Catherine was extremely indignant on the marriage of her nephew due to the abusive language she sent him in her...
58,The Life of Julius Caesar,What was Bartleby's demeanor like when he first appeared in the law-chambers?,Bartleby's demeanor was described as 'cadaverously gentlemanly _nonchalance_'.


## 4) Retrieval evaluation and ablation

These scores need no LLM, so it's cheap to compare many configurations. Each row adds one change.

* **Hit@k:** share of answerable questions where a chunk containing the gold evidence is in the top k. The generator sees `top_k = 4` chunks, so Hit@3 to Hit@5 is the number that matters most.
* **MRR / nDCG@10:** how high the first relevant chunk is ranked.
* **book_hit@k:** a looser check: did the right *book* show up at all?
* **abstain_auroc:** probability that a random answerable question gets a higher retrieval score than a random unanswerable one. 1.0 means a single threshold separates them perfectly.

In [6]:
factory = IndexFactory(books, cache_dir="emb_cache")   # shares models + embeddings across configs
ablation, ablation_items = compare_retrieval_configs(factory, default_ablation_configs(), test)
ablation[["hit@1", "hit@3", "hit@5", "hit@10", "book_hit@5", "mrr", "ndcg10", "abstain_auroc", "latency_p50_ms"]].round(3)

W1116 12:03:40.114000 4495 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

v1 baseline (MiniLM dense)       hit@1=0.21 hit@5=0.40 MRR=0.286 nDCG@10=0.330 gate-AUROC=0.96 (5s)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

+ BGE-small embeddings           hit@1=0.28 hit@5=0.59 MRR=0.395 nDCG@10=0.447 gate-AUROC=0.86 (1s)
+ contextual headers             hit@1=0.33 hit@5=0.57 MRR=0.430 nDCG@10=0.476 gate-AUROC=0.88 (1s)
BM25 only (+headers)             hit@1=0.43 hit@5=0.62 MRR=0.513 nDCG@10=0.547 gate-AUROC=0.88 (1s)
+ hybrid (dense+BM25, RRF)       hit@1=0.40 hit@5=0.66 MRR=0.527 nDCG@10=0.573 gate-AUROC=0.88 (2s)


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

+ cross-encoder rerank           hit@1=0.52 hit@5=0.72 MRR=0.615 nDCG@10=0.652 gate-AUROC=0.89 (43s)
+ title routing (full v2)        hit@1=0.52 hit@5=0.71 MRR=0.614 nDCG@10=0.651 gate-AUROC=0.89 (40s)


,hit@1,hit@3,hit@5,hit@10,book_hit@5,mrr,ndcg10,abstain_auroc,latency_p50_ms
config,,,,,,,,,
v1 baseline (MiniLM dense),0.207,0.345,0.397,0.483,0.931,0.286,0.330,0.955,3.577
+ BGE-small embeddings,0.276,0.500,0.586,0.603,0.948,0.395,0.447,0.859,5.323
+ contextual headers,0.328,0.500,0.569,0.638,0.931,0.430,0.476,0.878,5.566
BM25 only (+headers),0.431,0.569,0.621,0.707,0.966,0.513,0.547,0.884,10.578
"+ hybrid (dense+BM25, RRF)",0.397,0.638,0.655,0.724,0.948,0.527,0.573,0.878,20.024
+ cross-encoder rerank,0.517,0.707,0.724,0.759,0.983,0.615,0.652,0.892,537.026
+ title routing (full v2),0.517,0.707,0.707,0.759,0.983,0.614,0.651,0.892,537.869


In [7]:
ablation.to_csv(f"{OUT_DIR}/retrieval_ablation.csv")

## 5) Calibrate the abstention threshold (dev split only)

The system refuses to answer when its best retrieval score is below a threshold. We choose the threshold with the best balanced accuracy (share of answerable questions answered + share of unanswerable ones refused) on **dev**, while still answering at least 90% of answerable dev questions. Tuning on the test set would make the results look better than they really are.

In [8]:
v1_index = factory.build(RAGConfig.baseline())   # MiniLM, dense, no headers/rerank/routing
v2_index = factory.build(RAGConfig())            # full v2

cal_v1 = calibrate_abstention(v1_index, dev)
cal_v2 = calibrate_abstention(v2_index, dev)
for name, cal in [("v1 (cosine)", cal_v1), ("v2 (reranker)", cal_v2)]:
    b = cal["best"]
    print(f"{name:14s} threshold={cal['threshold']:.3f}  dev AUROC={cal['auroc']:.2f}  "
          f"answers {b['answer_coverage']:.0%} of answerable, refuses {b['negative_refusal']:.0%} of unanswerable")

v1 (cosine)    threshold=0.443  dev AUROC=0.95  answers 100% of answerable, refuses 80% of unanswerable
v2 (reranker)  threshold=0.068  dev AUROC=1.00  answers 100% of answerable, refuses 100% of unanswerable


## 6) End-to-end evaluation on the test split

Three systems are compared:
1. **v1 retrieval:** v1's retriever with a fairly calibrated cosine gate (generation bug fixed, otherwise it can't be scored at all)
2. **v2 without gate:** v2 retrieval, relying only on the prompt to say "I don't know"
3. **v2:** v2 retrieval with the calibrated re-ranker gate

For each answerable question, the judge grades correctness against the reference (1-5; 4 or above counts as correct). It also splits the answer into individual claims and checks each against the retrieved context. Negatives count as handled when the system refuses, or, for false-premise questions, when it doesn't go along with the premise.

In [9]:
judge = LLMJudge(llm)               # or LLMJudge(other_llm) for a different judge model
sim_model = factory._model("embed", "BAAI/bge-small-en-v1.5")

systems = {
    "v1 retrieval": RAGPipeline(v1_index, llm, RAGConfig.baseline(abstain_threshold=cal_v1["threshold"])),
    "v2 (no gate)": RAGPipeline(v2_index, llm, RAGConfig(abstain_threshold=None)),
    "v2":           RAGPipeline(v2_index, llm, RAGConfig(abstain_threshold=cal_v2["threshold"])),
}
summaries, item_results = {}, {}
for name, pipe in systems.items():
    print(f"\n===== {name} =====")
    summaries[name], item_results[name] = evaluate_end_to_end(pipe, test, judge=judge, embedder=sim_model)
    save_results(f"{OUT_DIR}/{name.replace(' ', '_').replace('(', '').replace(')', '')}", summaries[name], item_results[name])
print("judge JSON parse failures:", judge.parse_failures)


===== v1 retrieval =====
[1/73] cur-02   false_refusal (gold retrieved) (0s) Which book has a narrator who was originally called Robinson
[2/73] cur-16   wrong: retrieval miss      (17s) At the start of Anna Karenina, why is the Oblonsky household
[3/73] cur-12   correct                    (26s) In Great Expectations, how did Pip come to be called Pip?
[4/73] cur-21   false_refusal (retrieval miss) (29s) How does A Tale of Two Cities describe the period in its fam
[5/73] cur-14   correct                    (35s) Who does Dorothy live with in Kansas?
[6/73] cur-04   correct                    (43s) What is the name of Long John Silver's parrot?
[7/73] cur-05   false_refusal (retrieval miss) (46s) In Treasure Island, what words were tattooed on the old capt
[8/73] cur-07   correct                    (54s) What was written on the label of the little bottle Alice fou
[9/73] cur-18   wrong: retrieval miss      (64s) Who is Sir Robert Perry?
[10/73] cur-06   correct                    (73s)

### Results

Each value is shown as `mean [95% bootstrap CI]`. If two systems' confidence intervals overlap a lot, the test set is too small to call the difference real. Add more curated or synthetic questions.

In [10]:
comparison_table(summaries)

,v1 retrieval,v2 (no gate),v2
metric,,,
End-to-end success (all questions) ↑,"0.548 [0.44, 0.66]","0.753 [0.64, 0.85]","0.726 [0.62, 0.82]"
Answer accuracy (judge >= 4/5) ↑,"0.448 [0.31, 0.59]","0.707 [0.59, 0.83]","0.672 [0.57, 0.79]"
Answer correctness (1-5) ↑,2.879,3.793,3.672
Token F1 vs reference ↑,"0.260 [0.20, 0.33]","0.404 [0.34, 0.46]","0.383 [0.32, 0.45]"
Faithfulness (supported claims) ↑,"1.000 [1.00, 1.00]","1.000 [1.00, 1.00]","1.000 [1.00, 1.00]"
Hallucination rate ↓,"0.000 [0.00, 0.00]","0.000 [0.00, 0.00]","0.000 [0.00, 0.00]"
Context recall (gold in top-k) ↑,"0.362 [0.24, 0.48]","0.707 [0.59, 0.81]","0.707 [0.59, 0.81]"
Citation precision ↑,0.439,0.704,0.706
False-refusal rate ↓,"0.293 [0.19, 0.41]","0.069 [0.02, 0.14]","0.121 [0.05, 0.21]"


In [11]:
# Success rate by question type (answerable: judged correct; negatives: handled correctly)
qtype_table(summaries).round(2)

system,v1 retrieval,v2,v2 (no gate)
qtype,,,
cross_book,0.00,1.00,1.00
explanatory,0.50,0.50,0.50
factoid,0.58,0.75,0.75
false_premise,1.00,1.00,1.00
gibberish,1.00,1.00,1.00
metadata,1.00,1.00,1.00
missing_book,1.00,1.00,1.00
out_of_domain,0.75,0.75,0.75
synthetic,0.40,0.64,0.69


In [12]:
# Where do the failures come from? (retrieval miss vs. hallucination vs. generation error vs. refusal)
pd.DataFrame({name: s["outcomes"] for name, s in summaries.items()}).fillna(0).astype(int)

,v1 retrieval,v2 (no gate),v2
correct,26,41,39
false_refusal (retrieval miss),15,3,4
wrong: retrieval miss,14,9,8
correct_refusal,14,14,14
false_refusal (gold retrieved),2,1,3
wrong: generation error,1,4,4
answered_unanswerable,1,1,1


## 7) Failure analysis

In [13]:
show_failures(item_results["v2"], "wrong", n=5)
show_failures(item_results["v2"], "false_refusal", n=3)
show_failures(item_results["v2"], "answered_unanswerable", n=3)

--- cur-12 [wrong: generation error] In Great Expectations, how did Pip come to be called Pip?
    reference: As an infant he could not say anything longer than "Pip" for his family name Pirrip and Christian name Philip, so he called himself Pip.
    answer:    Pip came to be called Pip because the young lady who met him at the house of Mr. Pumblechook, upon being introduced to him, said "This is Pip," [2]. His conductor, Mr. Pumblechook, had earlier referred to him as Pip, [1].
    retrieved: ['Great Expectations @0', 'Great Expectations @108518', 'Great Expectations @454376', 'Great Expectations @275056']
--- cur-21 [wrong: retrieval miss] How does A Tale of Two Cities describe the period in its famous opening line?
    reference: "It was the best of times, it was the worst of times" - an age of wisdom and of foolishness, of belief and of incredulity.
    answer:    The period in "A Tale of Two Cities" is described as a time of retribution and change, with the old oppressors perishin

## 8) Validate the judge with a human spot check

LLM-judge scores are only useful if they agree with people. Label about 30 answers by hand in the exported CSV (`1` = correct, `0` = not), then compute agreement. A Cohen's kappa above about 0.6 means the judge's numbers are reasonable to report.

In [14]:
export_for_human_review(item_results["v2"], f"{OUT_DIR}/human_review.csv", n=30)
# ...after filling in the last column:
# judge_human_agreement(f"{OUT_DIR}/human_review.csv")

Wrote 30 rows to eval_outputs/human_review.csv; fill in the last column, then call judge_human_agreement().


## 9) Try it

In [16]:
rag = systems["v2"]

def ask(question):
    out = rag.answer(question)
    print(f"Q: {question}\nA: {out['answer']}\n")
    if not out["abstained_by_gate"]:
        for i, h in enumerate(out["hits"], 1):
            print(f"  [{i}] {h.chunk.title} (chars {h.chunk.start}-{h.chunk.end}) rerank={h.rerank:.2f}")
    print(f"  gate score={out['gate_score']:.3f}  latency={out['timings']['total_s']:.1f}s")

ask("Which book has the character Robinson Kreutznaer?")
ask("What does the parrot keep screaming in Treasure Island?")
ask("explain me pythagorean theorem?")

Q: Which book has the character Robinson Kreutznaer?
A: The character Robinson Kreutznaer is from the book "Robinson Crusoe" by Daniel Defoe [1].

  [1] Robinson Crusoe (chars 0-920) rerank=0.86
  [2] Moby Dick (chars 1054374-1054558) rerank=0.01
  [3] The Lord of the Rings (chars 7836-8187) rerank=0.00
  [4] Moby Dick (chars 167121-167213) rerank=0.00
  gate score=0.861  latency=2.5s
Q: What does the parrot keep screaming in Treasure Island?
A: The parrot in Treasure Island keeps screaming "Pieces of eight!" [2]

  [1] Treasure Island (chars 100939-101523) rerank=0.81
  [2] Treasure Island (chars 100079-100986) rerank=0.77
  [3] Treasure Island (chars 99217-100076) rerank=0.38
  [4] Treasure Island (chars 341283-342107) rerank=0.23
  gate score=0.813  latency=1.6s
Q: explain me pythagorean theorem?
A: I don't know based on the available books.

  gate score=0.005  latency=0.6s


In [17]:
# The same pipeline as a LangGraph graph (retrieve -> gate -> generate | abstain)
graph = build_langgraph(rag)
graph.invoke({"question": "Who painted the portrait of Dorian Gray?"})["answer"]

'The portrait of Dorian Gray was painted by Basil Hallward [4].'

## Caveats

* **The judge grading its own model:** Mistral judging Mistral tends to be lenient. Use a different judge model and check against human labels (section 8).
* **Synthetic questions** come from single chunks, so they favor "find one passage" lookups. The curated set and the per-type table show how the system does on cross-book, metadata ("which books are by Dickens?"), and false-premise questions. Those are harder and closer to what real users ask.
* **Sample size:** with about 70 test questions, a 5-point difference is usually inside the confidence interval. Treat small differences as noise.